# Comprehensive Analysis Of Hotel Booking Patterens

Group Members
1. Rafid Islam
2. Umme Muskan
3. Sadia Sanji
4. Sadia Rumpa

# Context
1. What one row represents: Each individual row in the dataset represents a unique hotel booking record.

2. Population and time period: The dataset comprises 1,000 documented booking records extracted from historical hotel management systems in Portugal.

3. Purpose of analysis: To investigate booking patterns, cancellation drivers, guest characteristics, pricing dynamics (average_daily_rate), and operational differences between city and resort hotels to provide actionable insights for hotel management.


# Research Questions
1. Do cancellation rates differ significantly between city hotels and resort hotels, and how do deposit types moderate this relationship?

2. How does booking lead time vary across customer types, and is longer lead time associated with a higher likelihood of cancellation?

3. How do booking volumes, average daily rates, and total guest counts fluctuate across different market segments and arrival months?


# Import & Display Settings

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 25)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: "%.2f" % x)
plt.style.use("seaborn-v0_8-whitegrid")

# Load & Inspect the Data


In [5]:
df_raw = pd.read_csv("hotel_bookings_final_project.csv")

print(f"Dataset Shape (Rows, Columns): {df_raw.shape}")
print("\n--- Column Data Types & Non-Null Counts ---")
df_raw.info()

print("\n--- Missing Values Summary ---")
missing = df_raw.isnull().sum()
print(missing[missing > 0])

print(f"\n--- Exact Duplicate Rows: {df_raw.duplicated().sum()} ---")

print("\n--- Random Sample (5 rows) ---")
display(df_raw.sample(5, random_state=42))

print("\n--- Summary Statistics (Numerical Variables) ---")
display(df_raw.describe())

Dataset Shape (Rows, Columns): (1000, 20)

--- Column Data Types & Non-Null Counts ---
<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   hotel                        1000 non-null   str    
 1   is_canceled                  1000 non-null   int64  
 2   lead_time                    1000 non-null   int64  
 3   arrival_date_year            1000 non-null   int64  
 4   arrival_date_month           1000 non-null   str    
 5   arrival_date_day_of_month    1000 non-null   int64  
 6   stays_in_weekend_nights      1000 non-null   int64  
 7   stays_in_week_nights         1000 non-null   int64  
 8   adults                       1000 non-null   int64  
 9   children                     1000 non-null   int64  
 10  babies                       1000 non-null   int64  
 11  country                      998 non-null    str    
 12  m

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,country,market_segment,deposit_type,customer_type,average_daily_rate,is_repeated_guest,booking_changes,required_car_parking_spaces,total_of_special_requests
521,City Hotel,0,190,2016,December,28,0,4,2,0,0,DEU,Direct,No Deposit,Transient,167.80,0,0,0,1
737,Resort Hotel,0,225,2016,May,26,0,3,1,0,0,GBR,Groups,No Deposit,Transient-Party,66.00,0,2,1,0
740,City Hotel,1,105,2017,June,26,1,3,2,0,0,CHE,Online TA,No Deposit,Transient,126.00,0,0,0,0
660,City Hotel,0,198,2017,March,17,1,2,2,0,1,NLD,Online TA,No Deposit,Transient,93.60,0,2,0,3
411,Resort Hotel,0,54,2016,January,9,2,2,2,0,0,GBR,Offline TA/TO,No Deposit,Transient,49.40,0,3,0,0



--- Summary Statistics (Numerical Variables) ---


,is_canceled,lead_time,arrival_date_year,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,average_daily_rate,is_repeated_guest,booking_changes,required_car_parking_spaces,total_of_special_requests
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,0.37,105.06,2016.13,16.13,0.92,2.52,1.86,0.11,0.01,101.21,0.03,0.24,0.06,0.59
std,0.48,106.45,0.70,8.96,0.95,1.80,0.53,0.40,0.09,45.72,0.18,0.67,0.24,0.80
min,0.00,0.00,2015.00,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,0.00,19.00,2016.00,8.00,0.00,1.00,2.00,0.00,0.00,68.57,0.00,0.00,0.00,0.00
50%,0.00,69.50,2016.00,16.00,1.00,2.00,2.00,0.00,0.00,94.50,0.00,0.00,0.00,0.00
75%,1.00,157.00,2017.00,24.00,2.00,3.00,2.00,0.00,0.00,125.00,0.00,0.00,0.00,1.00
max,1.00,542.00,2017.00,31.00,8.00,20.00,6.00,3.00,1.00,304.00,1.00,7.00,1.00,5.00


# Data-quality assessment and cleaning log

| Issue | Checked Evidence Found | Decision | Reason |
| :--- | :--- | :--- | :--- |
| **Missing Values** | `country` has 2 missing entries; structural columns (`agent`, `company`) have missing values. | Keep / Fill | `country` missing values filled with 'PRT'; structural column NaNs left intact as they denote direct/independent bookings. |
| **Exact Duplicates** | 26 identical booking rows identified. | Remove | Eliminates double-counting distortion in booking counts and frequency distributions. |
| **Impossible Negative Values** | 0 negative values found in `lead_time` or `average_daily_rate`; 1 record had ADR = 0.00. | Retain / Inspect | ADR of 0.00 corresponds to promotional/complimentary stays rather than data corruption. |
| **Zero Total Guests** | 3 bookings where adults + children + babies == 0. | Remove | Physically impossible booking record; cannot have zero occupants. |
| **Zero Total Nights** | 2 bookings where weekend nights + weekday nights == 0. | Remove | Invalid reservation length; minimum stay must be at least 1 night. |
| **Unusual Lead Times** | Maximum `lead_time` reaches 542 days. | Retain | Valid long-term advance booking behavior for resort properties. |

# Transformations and derived variables

# Individual-Variable analysis


Categorical Variables Analysis:
1. Hotel Type: City Hotel ($n = 614$, $63.36\%$), Resort Hotel ($n = 355$, $36.64\%$).
2. Deposit Type: No Deposit ($n = 865$, $89.27\%$), Non Refund ($n = 101$, $10.42\%$), Refundable ($n = 3$, $0.31\%$).

Numerical Variables Analysis:
1. Lead Time ($n = 969$): Mean = $103.30$ days, Median = $68.00$ days, Std = $104.88$ days, Min = $0.00$ days, Max = $542.00$ days.
2. Average Daily Rate ($n = 969$): Mean = $\$101.97$, Median = $\$94.50$, Std = $\$45.68$, Min = $\$0.00$, Max = $\$304.00$.

# Relationships and grouped analysis

# Final Visualizations

# Conclusions, limitations, and recommendation

# Exports and verification